# **Procesamiento del Lenguaje Natural**
## *Práctica final - Detección de lenguaje ofensivo en redes sociales*

## Recursos a importar

In [ ]:
# En caso de estar trabajando con Google Colab
# from google.colab import drive
# drive.mount("/content/drive")

In [ ]:
!pip install pandas
!pip install numpy==1.23.5

In [ ]:
# NLTK
!pip install nltk
import nltk

nltk.download("punkt")

# Corpus de palabras vacias
from nltk.corpus import stopwords

nltk.download("stopwords")
spanish_stops = stopwords.words("spanish")

from nltk.tokenize import WordPunctTokenizer

# Reducir palabras a su raíz
from nltk.stem.snowball import SnowballStemmer

stemmer = SnowballStemmer("spanish")

In [ ]:
!pip install spacy
# Análisis morfológico
import spacy.cli

spacy.cli.download("es_core_news_sm")
import es_core_news_sm

nlp = es_core_news_sm.load()

In [ ]:
!pip install scikit-learn

# Vector de características
from sklearn.feature_extraction.text import TfidfVectorizer

# Modelo SVM
from sklearn import svm


In [ ]:
# Recuros extras
import os
import csv
import re

## Variables Globales
Definimos una serie de variables globales que serán compartidas por diferentes funciones

In [ ]:
# Variables globales
model = None
vectorizer = None
lista_comentarios_entrenar: list = []
lista_comentarios_test: list = []
lexicon : set = set()

# Quitamos la palabra no de los stopWords
if('no' in spanish_stops):
  spanish_stops.remove('no')

# Clase de comentario
Los atributos de la clase son:
* **comment_id**: Identificador del comentario
* **comment**: Comentario
* **label**: Etiqueta con la que se ha clasificado
* **influencer_gender**: Género de la persona a la que va dirigido el comentario
* **media**: Plataforma en la que se ha publicado el mensaje

In [ ]:
class Comentario:
  def __init__(self, comment_id, comment, label, influencer_gender, media):
    self.comment_id = comment_id
    self.comment = comment
    self.label = label
    self.influencer_gender = influencer_gender
    self.media = media

## Función de preocesar texto
Función que recibe como parámetros:
* **texto** : Texto que se quiere procesar.  

Los pasos de la función son:
  1. Pasar el texto a minúscula, quitar los signos de puntuación y las tildes
  2. Tokenizar el texto, extrayendo las palabras
  3. Quitar las palabras vacías
  4. Obtener las raíces de cada palabra/token extraído
  5. Devolver el texto procesado

In [ ]:
def procesar_texto(texto):
  tk = WordPunctTokenizer() # Tokenizador a utilizar

  # Quitamos los signos de puntuación del texto y pasamos a minúscula
  texto_puntuacion = re.sub(r'[^\w\s]','',texto.lower())
  # Quitamos las tildes del comentario
  texto_puntuacion = texto_puntuacion.translate(texto_puntuacion.maketrans("áàäéèëíìïòóöùúü", "aaaeeeiiiooouuu"))
  # Tokenizamos el texto
  palabras = tk.tokenize(str(texto_puntuacion))

  # Quitamos las palabras vacías
  palabras_no_vacias = [word for word in palabras if word not in spanish_stops]

  # Reducimos las palabras de toda la lista a su raíz
  lista_raices = []
  for palabra in palabras_no_vacias:
    lista_raices.append(stemmer.stem(palabra))
  
  # Devolvemos una cadena con el texto procesado
  return " ".join(lista_raices)

## Función para obtener contenido TSV y almacenarlo en una lista
Función que recibe como parámetros:
* **rutaArchivo** : Ruta del archivo excel que contiene los datos
* **listaArchivos**: Lista donde se van a almacenar los comentarios

Los pasos de la función son:
  1. Abrimos el fichero que contiene los comentarios
  2. Extraemos cada comentario línea a línea y lo almacenamos en la lista pasada

In [ ]:
def obtener_documentos_tsv(ruta_archivo, lista_archivos):
  # Comprobamos si existe el archivo pasado
  try:
    # Abrimos el archivo 
    f = open(ruta_archivo, "r", errors="ignore")

    # Separamos cada línea
    read_tsv = csv.reader(f, delimiter="\t")
    
    for r in read_tsv:
      # Leemos todas las lineas menos la primera que son los títulos
      comentario = Comentario(r[0],r[1],r[2],r[3],r[4])

      lista_archivos.append(comentario)

    # Eliminamos la primera línea que contiene los títulos
    lista_archivos.pop(0)
    print("El número de comentarios es de: " + str(len(lista_archivos)))
    f.close()
  except FileNotFoundError:
    print("No existe el archivo " + ruta_archivo)

## Función para obtener el lexicón
Función que recibe como parámetros:
* **rutaArchivo**: Ruta del archivo *.txt*, que contiene el lexicón

Los pasos a realizar por la función son:
1. Abrir el fichero pasado como parámetro
2. Leer cada línea del fichero
    1. Procesar cada línea del fichero
    2. Añadir cada línea a la variable global del lexicón

In [ ]:
def obtener_lexicon(ruta_archivo):
  try:
    # Abrimos el archivo si es de extensión txt
        _, extension = os.path.splitext(ruta_archivo)
        if(extension == '.txt'):
          with open(ruta_archivo, "r", errors="ignore",encoding='utf-8-sig') as f:
            # Leemos cada línea/palabra del fichero
            linea_leida = f.readline()
            while(linea_leida):
              linea_procesada = procesar_texto(linea_leida)
              # Separamos la palabra por espacios
              array_palabras = linea_procesada.split()

              linea = ""
              i = 0
              while(i < len(array_palabras)):
                linea += array_palabras[i]
                i += 1
                if(i < len(array_palabras)):
                  linea += " "

              global lexicon
              lexicon.add(linea)
              linea_leida = f.readline()
     
          print("El número de palabras en el lexicón es de: " + str(len(lexicon)))
  except FileNotFoundError:
    print("No existe el archivo " + ruta_archivo)

## Función para entrenar el modelo
La función realizará los siguientes pasos:
1. Obtenemos el vector de características del los textos de entrenamiento.
2. Creamos el modelo SVM
3. Obtenemos todos los comentarios de entrenamiento y los procesamos
4. Obtenemos todos los términos del lexicón de insultos
5. Entrenamos el modelo con la función **.fit**

In [ ]:
def entrenar_modelo():
    # Obtenemos la lista del texto de comentarios
    lista_documentos: list = []
    tags: list = []

    # Obtenemos todos los comentarios de entrenamiento
    global lista_comentarios_entrenar
    for c in lista_comentarios_entrenar:
        comentario_procesado = procesar_texto(c.comment)
        lista_documentos.append(comentario_procesado)
        tags.append(c.label)

    # Obtenemos el vector de características del los textos de entrenamiento que se han pasado
    global vectorizer
    vectorizer = TfidfVectorizer()
    X = vectorizer.fit_transform(lista_documentos)
    # Creamos el modelo con la clase SVC
    global model
    model = svm.SVC()
    model.fit(X, tags) # Entrenamiento

## Función para predecir un texto
Función que recibe como parámetros:
* **documento** : Texto que se quiere predecir

La función realiza los siguientes pasos:
1. Procesamos el texto y lo almacenamos en una variable local.
2. Obtenemos el vector de características del texto a predecir
3. Predecimos a la etiqueta que pertenecerá el texto procesado

In [ ]:
def predecir_comentario(documento):
  # Procesamos el texto pasado
  contenido = [procesar_texto(documento)]
  
  # Obtenemos la predicción con el aprendizaje supervisado
  global vectorizer
  Y = vectorizer.transform(contenido)

  global model
  prediction = model.predict(Y)

  # Devolemos la predicción
  return prediction[0]

## Prediccion de comentarios
Función que recibe como parámetros
* **ruta_fichero_prediccion**: Ruta del fichero que almacenará los resultados de la predicción de la lista de comentarios para test  

La función realiza los siguientes pasos:
1. Crear el fichero en la ruta indicada como parámetro
2. Recorrer toda la lista de los comentarios de test
    1. Predecimos la categoría para cada comentario
    2. Escribimos el resultado en el fichero

In [ ]:
def predecir_comentarios(ruta_fichero_prediccion):
    global lista_comentarios_test
    # Comprobamos que hay al menos un comentario en la lista de test
    if len(lista_comentarios_test) > 0:
        # Creamos un fichero
        fichero_prediccion = open(ruta_fichero_prediccion, "w")  # si existía el fichero, ha borrado su contenido
        fichero_prediccion.write("comment_id\tpred\n")

        # Recorremos toda la lista de comentarios de test
        for c in lista_comentarios_test:
            # Obtenemos el contenido del comentario
            prediccion = predecir_comentario(c.comment)
            # Guardamos el resultado en el fichero
            linea = str(c.comment_id) + "\t" + str(prediccion) + "\n"
            fichero_prediccion.write(linea)

        # Cerramos el fichero
        fichero_prediccion.close()

## Función para obtener el porcentaje de insultos a cada género
Función que recibe como parámetros:
* **listaComentario**: Lista de comentarios
* **rutaFicheroPrediccion**: Ruta del fichero de la predicción

In [ ]:
def porcentaje_genero(lista_comentarios,ruta_fichero_prediccion):
  man = 0
  woman = 0

  # Pasamos la lista de comentarios a diccionario por ID-GENERO
  diccionario_genero: dict = {}
  for c in lista_comentarios:
    diccionario_genero[c.comment_id] = c.influencer_gender

  # Abrimos el fichero de comentarios predecidos
  try:
    # Abrimos el archivo 
    f = open(ruta_fichero_prediccion, "r", errors="ignore")

    # Separamos cada línea
    read_tsv = csv.reader(f, delimiter="\t")
    
    # Leemos todas las lineas 
    for r in read_tsv:
      # Comprobamos si el mensaje es ofensivo
      if(r[1] == "OFF"):
        # Buscamos en la lista de comentarios si se trata de un hombre o una mujer
        if(diccionario_genero[r[0]] == "man"):
          man += 1
        else:
          woman += 1

    # Mostramos los resultados
    print("El porcentaje de mensajes ofensivos a hombres(man): " + str((man * 100) / (man + woman)) + " %" )
    print("El porcentaje de mensajes ofensivos a mujeres(woman): " + str((woman * 100) / (man + woman)) + " %" )

  except FileNotFoundError:
    print("No existe el archivo " + str(ruta_fichero_prediccion))

## Función para obtener el porcentaje de insultos en cada plataforma
Función que recibe como parámetros:
* **listaComentario**: Lista de comentarios
* **rutaFicheroPrediccion**: Ruta del fichero de la predicción

In [ ]:
def porcentaje_plataforma(lista_comentarios,ruta_fichero_prediccion):
  conjunto_platafromas: set = set()
  diccionario_final: dict = {}

  # Pasamos la lista de comentarios a diccionario por ID-Plataforma
  diccionario_plataforma: dict = {}
  for c in lista_comentarios:
    diccionario_plataforma[c.comment_id] = c.media
    if(c.media not in conjunto_platafromas):
      conjunto_platafromas.add(c.media)
      diccionario_final[c.media] = 0

  # Abrimos el fichero de comentarios predecidos
  try:
    # Abrimos el archivo 
    f = open(ruta_fichero_prediccion, "r", errors="ignore")

    # Separamos cada línea
    read_tsv = csv.reader(f, delimiter="\t")
    
    # Leemos todas las lineas
    total = 0
    for r in read_tsv:
      # Comprobamos si el mensaje es ofensivo
      if(r[1] == "OFF"):
        # Buscamos en la lista de comentarios para saber a que plataforma pertenece
        plataforma = diccionario_plataforma[r[0]]
        diccionario_final[plataforma] += 1
        total += 1
    
    # Mostramos los resultados para cada plataforma
    for p in conjunto_platafromas:
      porcentaje = diccionario_final[p] * 100 / total
      print("El porcentaje de mensajes ofensivos en la plataforma " + p + " es de: " + str(porcentaje) + " %" )

  except FileNotFoundError:
    print("No existe el archivo " + str(ruta_fichero_prediccion))

## Ejecución del ejercicio
Para ejecutar el ejercicio, primero deberemos limpiar la lista de documentos y categorias y asignar a nulo el modelo y vector a entrenar.

In [ ]:
# Obtenemos los comentarios de la lista de entrenamiento
lista_comentarios_entrenar.clear()
obtener_documentos_tsv("Material/train.tsv",lista_comentarios_entrenar)
# Obtenemos los comentarios de la lista de test
lista_comentarios_test.clear()
obtener_documentos_tsv("Material/test.tsv",lista_comentarios_test)
# Obtenemos el lexicon
lexicon.clear()
obtener_lexicon("Material/SHARE.txt")
# Entrenamos el modelo
model = None
vectorizer = None
entrenar_modelo()

In [ ]:
# Predecimos el test
predecir_comentarios("Material/prediction_test.tsv")

In [ ]:
porcentaje_genero(lista_comentarios_test,"Material/prediction_test.tsv")
print()
porcentaje_plataforma(lista_comentarios_test,"Material/prediction_test.tsv")

## Fichero generado por el script scorer.py
Métricas que se utilizan:

1. **Precisión**:Con la métrica de precisión podemos medir la calidad del modelo de machine learning en tareas de clasificación.
2. **Recall**: La métrica de exhaustividad nos va a informar sobre la cantidad que el modelo de machine learning es capaz de identificar.
3. **F1_SCORE**: El valor F1 se utiliza para combinar las medidas de precision y recall en un sólo valor. Esto es práctico porque hace más fácil el poder comparar el rendimiento combinado de la precisión y la exhaustividad entre varias soluciones.


In [ ]:
!python3 Material/scorer.py Material/gold_label_test.tsv Material/prediction_test.tsv